# 05_tuning_lightgbm.ipynb

## 목적
04_baseline_comparison에서 속도와 성능을 모두 고려해 선정된 **LightGBM**를 정밀 튜닝합니다.
GridSearchCV 탐색 범위를 베이스라인보다 넓혀 추가 성능 개선을 시도합니다.

## 목차
1. 환경 세팅 및 데이터 로드
2. 하이퍼파라미터 튜닝 (GridSearchCV)
3. 최종 모델 학습 및 평가
4. 피처 중요도 분석
5. 베이스라인 대비 개선 효과 확인
6. 결과 저장


## 1. 환경 세팅 및 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import time, os

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, roc_curve, classification_report
)

plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.4f}'.format
print('임포트 완료')


임포트 완료


In [2]:
X_train = pd.read_csv('../data/X_train_v2.csv')
X_val   = pd.read_csv('../data/X_val_v2.csv')
y_train = pd.read_csv('../data/y_train_v2.csv').squeeze()
y_val   = pd.read_csv('../data/y_val_v2.csv').squeeze()

# 04_baseline_comparison에서 측정된 베이스라인 성능 (비교용)
baseline_auc = pd.read_csv('../results/baseline_comparison_v2.csv')
baseline_row = baseline_auc[baseline_auc['model'] == 'LightGBM']
baseline_auc_score = baseline_row['auc_roc'].values[0]

print(f'X_train: {X_train.shape}, X_val: {X_val.shape}')
print(f'LightGBM 베이스라인 AUC: {baseline_auc_score}')


X_train: (914921, 50), X_val: (228731, 50)
LightGBM 베이스라인 AUC: 0.8477


## 2. 하이퍼파라미터 튜닝 (GridSearchCV)

In [3]:
# 너무 오래걸려서 랜덤서치 먼저 시도

from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    'num_leaves'       : [31, 63, 127, 255],
    'max_depth'        : [-1, 10, 15, 20],
    'learning_rate'    : [0.03, 0.05, 0.1],
    'n_estimators'     : [200, 300, 400],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.7, 0.8, 0.9],
    'min_child_samples': [20, 50, 100]
}

base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

grid_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_grid,
    n_iter=20,          # 3,888개 조합 중 20개만 무작위로 시도
    scoring='roc_auc',
    cv=2,                # 3 → 2로 축소 (검증 fold 수 줄이기)
    n_jobs=-1,
    random_state=42,
    verbose=1
)

start = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start

best_params = grid_search.best_params_
print(f'튜닝 소요 시간: {tuning_time:.1f}초')
print('--- 최적 파라미터 ---')
print(best_params)
print(f'CV 최고 AUC: {grid_search.best_score_:.4f}')

Fitting 2 folds for each of 20 candidates, totalling 40 fits
튜닝 소요 시간: 576.7초
--- 최적 파라미터 ---
{'subsample': 0.7, 'num_leaves': 127, 'n_estimators': 300, 'min_child_samples': 20, 'max_depth': 20, 'learning_rate': 0.03, 'colsample_bytree': 0.8}
CV 최고 AUC: 0.8477


In [ ]:
param_grid = {
    'num_leaves'       : [31, 63, 127, 255],
    'max_depth'        : [-1, 10, 15, 20],
    'learning_rate'    : [0.03, 0.05, 0.1],
    'n_estimators'     : [200, 300, 400],
    'subsample'        : [0.7, 0.8, 0.9],
    'colsample_bytree' : [0.7, 0.8, 0.9],
    'min_child_samples': [20, 50, 100]
}

base_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

grid_search = GridSearchCV(
    estimator=base_model, param_grid=param_grid,
    scoring='roc_auc', cv=3, n_jobs=-1, verbose=1
)

start = time.time()
grid_search.fit(X_train, y_train)
tuning_time = time.time() - start

best_params = grid_search.best_params_
print(f'튜닝 소요 시간: {tuning_time:.1f}초')
print('--- 최적 파라미터 ---')
print(best_params)
print(f'CV 최고 AUC: {grid_search.best_score_:.4f}')


## 3. 최종 모델 학습 및 평가

In [ ]:
final_model = lgb.LGBMClassifier(**best_params, random_state=42, n_jobs=-1, verbose=-1)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)

start = time.time()
# 위 fit 호출은 이미 실행됨 — 시간 측정용 재기록
train_time = time.time() - start


In [ ]:
y_pred  = final_model.predict(X_val)
y_proba = final_model.predict_proba(X_val)[:, 1]

final_acc = accuracy_score(y_val, y_pred)
final_auc = roc_auc_score(y_val, y_proba)
final_f1  = f1_score(y_val, y_pred)

print('--- LightGBM 튜닝 후 최종 성능 ---')
print(f'Accuracy : {final_acc:.4f}')
print(f'AUC-ROC  : {final_auc:.4f}')
print(f'F1-Score : {final_f1:.4f}')
print()
print(classification_report(y_val, y_pred, target_names=['유지(0)','이탈(1)']))


In [ ]:
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['유지(0)','이탈(1)'], yticklabels=['유지(0)','이탈(1)'])
plt.title('LightGBM (Tuned) Confusion Matrix')
plt.ylabel('실제값'); plt.xlabel('예측값')
plt.tight_layout(); plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'LightGBM Tuned (AUC = {final_auc:.4f})', color='steelblue', lw=2)
plt.plot([0,1],[0,1], linestyle='--', color='gray', label='Random Baseline')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('LightGBM (Tuned) ROC Curve')
plt.legend(); plt.tight_layout(); plt.show()


## 4. 피처 중요도 분석

In [ ]:
importance_df = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(x='importance', y='feature', data=importance_df, palette='Blues_r')
plt.title('LightGBM (Tuned) Feature Importance (Top 20)')
plt.tight_layout(); plt.show()

new_features = ['freq_decay_rate','recent_price_ratio',
                'unique_articles_last_90d','diversity_decay_rate',
                'last_interval','interval_acceleration']
all_imp = pd.DataFrame({
    'feature'   : X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)
all_imp['전체 순위'] = all_imp.index + 1

print('--- [NEW] 신규 피처 중요도 순위 ---')
print(all_imp[all_imp['feature'].isin(new_features)][['전체 순위','feature','importance']].to_string(index=False))


## 5. 베이스라인 대비 개선 효과 확인

In [ ]:
improvement = final_auc - baseline_auc_score

print('--- LightGBM 베이스라인 vs 튜닝 비교 ---')
print(f'베이스라인 AUC : {baseline_auc_score:.4f}')
print(f'튜닝 후 AUC    : {final_auc:.4f}')
print(f'개선폭         : {improvement:+.4f}')

if improvement < 0.005:
    print('\n→ 튜닝 효과가 미미합니다. 기본 파라미터가 이미 데이터에 적합했을 가능성이 높습니다.')
else:
    print('\n→ 튜닝으로 유의미한 성능 개선을 확인했습니다.')


## 6. 결과 저장

In [ ]:
os.makedirs('./results', exist_ok=True)

result = pd.DataFrame([{
    'model'        : 'LightGBM_tuned',
    'accuracy'     : round(final_acc, 4),
    'auc_roc'      : round(final_auc, 4),
    'f1_score'     : round(final_f1, 4),
    'baseline_auc' : round(baseline_auc_score, 4),
    'improvement'  : round(improvement, 4)
}])

result.to_csv('./results/lightgbm_tuned_result.csv', index=False)
print('결과 저장 완료')
print(result.to_string(index=False))
